In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path

# Assume this notebook is in project_root/scripts/runner.ipynb
project_root = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parents[0]
sys.path.append(str(project_root / "src"))
sys.path.append(str(project_root / "config"))

from pipeline.synthea_runner import SyntheaRunner
from pipeline.bq_loader import BigQueryLoader
from pipeline.bq_transformer import BigQueryTransformer
from pipeline.dictionary_builder import DictionaryBuilder
from pipeline.preprocessing import DataPreprocessor
from pipeline.model_config_manager import ModelConfigManager
from pipeline.hyperparameter_tuner import HyperparameterTuner
from pipeline.model_registry import ModelRegistry
from pipeline.evaluator import Evaluator
from pipeline.cost_reducer import CostReducer


logging.basicConfig(level=logging.INFO)
X : dict [str, pd.DataFrame()] = {}
y : dict [str, pd.DataFrame()] = {}

In [2]:
"""Initialize ModelConfigManager, populate models, initialize HyperparameterTuner and tune hyperparams"""
config_path_bq = str(project_root / "config" / "bigquery_config.json")
model_config_path = str(project_root / "config" / "model_config.json")
recipe_path = str(project_root / "config" / "bigquery_recipes.json")
cost_config_path = str(project_root / "config" / "cost_config.json")

cfg_mgr = ModelConfigManager.from_config(model_config_path)
active_models = cfg_mgr.list_active_models()
registry = ModelRegistry.from_config(model_config_path)
evaluator = Evaluator(registry=registry, cfg_mgr=cfg_mgr)
preproc = DataPreprocessor.from_config(model_config_path)
cost_reducer = CostReducer.from_config(cost_config_path)
tuner = HyperparameterTuner(config_mgr=cfg_mgr, target_col='readmit_30d')

recipes_id = 2

for profile_name in ['train', 'test']:

    transformer, t_cfg = BigQueryTransformer.from_profile(
        config_path=config_path_bq,
        profile_name=profile_name,
    )

    """Load and preprocess index stays"""
    X[profile_name], y[profile_name] = preproc.load_and_preprocess(
        transformer=transformer,
        force_query=False
    )

#tuner.tune_models(X['train'], y['train'], cost_config_path=cost_config_path)
#cfg_mgr.save()

fitted_models = registry.fit_models(
    X=X['train'],
    y=y['train'],
    target_cols=["readmit_30d"],
    model_names=None  # all active models from config
)

eval_results = evaluator.evaluate_models(
    X=X['test'],
    y=y['test'],
    model_names=None        # all active   
)

eval_results.update(evaluator.build_threshold_metrics(eval_results["pred_values"]))

mapping, avoided, pct_avoided = cost_reducer.map_estimate_cost_reduction(
    pred_values=eval_results['pred_values'],
    df_thresholds=eval_results['thresholds'],
)

report = evaluator.build_performance_report(
    pct_avoided,
    avoided,
    eval_results['threshold_metrics'],
    dataset_end_date=pd.Timestamp("2026-02-16")
)

INFO:pipeline.bq_transformer:Dataset already exists: hospital-readmission-4.data_slim
INFO:pipeline.bq_transformer:Dataset already exists: hospital-readmission-4.helper_tables
INFO:pipeline.bq_transformer:Dataset already exists: hospital-readmission-4.data_slim
INFO:pipeline.bq_transformer:Dataset already exists: hospital-readmission-4.helper_tables
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[LightGBM] [Info] Number of positive: 1967, number of negative: 197565
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008842 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1447
[LightGBM] [Info] Number of data points in the train set: 199532, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.009858 -> initscore=-4.609558
[LightGBM] [Info] Start training from score -4.609558
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_d

DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\logreg_d30.pkl exists: True
gets in
DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\rf_d30.pkl exists: True
gets in
DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\lightgbm_d30.pkl exists: True
gets in


In [2]:
"""shortcut for report"""
model_config_path = str(project_root / "config" / "model_config.json")
cfg_mgr = ModelConfigManager.from_config(model_config_path)
active_models = cfg_mgr.list_active_models()
registry = ModelRegistry.from_config(model_config_path)
evaluator = Evaluator(registry=registry, cfg_mgr=cfg_mgr)

pct_avoided = pd.read_csv(r"D:\Python Projects\Hospital readmission risk\scripts\data\artifacts\pct_avoided.csv")
avoided = pd.read_csv(r"D:\Python Projects\Hospital readmission risk\scripts\data\artifacts\avoided.csv")
threshold_metrics = pd.read_csv(r"D:\Python Projects\Hospital readmission risk\scripts\data\artifacts\threshold_metrics.csv").set_index("Unnamed: 0")
report = evaluator.build_performance_report(
    pct_avoided,
    avoided,
    threshold_metrics,
    dataset_end_date=pd.to_datetime("2026-02-16")
)


DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\logreg_d30.pkl exists: True
gets in
DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\rf_d30.pkl exists: True
gets in
DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\lightgbm_d30.pkl exists: True
gets in


In [2]:
model_config_path = str(project_root / "config" / "model_config.json")
cfg_mgr = ModelConfigManager.from_config(model_config_path)
active_models = cfg_mgr.list_active_models()
registry = ModelRegistry.from_config(model_config_path)
evaluator = Evaluator(registry=registry, cfg_mgr=cfg_mgr)

pct_avoided = pd.read_csv(r"D:\Python Projects\Hospital readmission risk\scripts\data\artifacts\pct_avoided_old.csv").set_index("Unnamed: 0")
avoided = pd.read_csv(r"D:\Python Projects\Hospital readmission risk\scripts\data\artifacts\avoided_old.csv").set_index("Unnamed: 0")
threshold_metrics = pd.read_csv(r"D:\Python Projects\Hospital readmission risk\scripts\data\artifacts\threshold_metrics_old.csv").set_index("Unnamed: 0")
report = evaluator.build_performance_report(
    pct_avoided,
    avoided,
    threshold_metrics,
    dataset_end_date=pd.to_datetime("2026-02-16"),
    suffix="old"
)

DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\logreg_d30_old.pkl exists: True
gets in
DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\rf_d30_old.pkl exists: True
gets in
DEBUG model_path: D:\Python Projects\Hospital readmission risk\.secrets\models\lightgbm_d30_old.pkl exists: True
gets in


In [5]:
pct_avoided.max(axis = 1, numeric_only = True)

Unnamed: 0
total_pct_avoided_0.2_0.1     0.026572
total_pct_avoided_0.2_0.15    0.067531
total_pct_avoided_0.2_0.2     0.065276
dtype: float64

In [3]:
report

,best_threshold,pct_cost_saved,cost_saved,train_date,dataset_end_date,tp,tn,fp,fn,precision,recall,f1
model_name,,,,,,,,,,,,
lightgbm_d30,0.95,-0.077192,-5.818943e+05,2026-03-18 12:07:17.213647604,2026-02-16,27.0,65720.0,83.0,659.0,0.245455,0.039359,0.067839
logreg_d30,0.95,-1.011114,-7.622089e+06,2026-03-18 11:56:12.729555607,2026-02-16,31.0,65591.0,212.0,655.0,0.127572,0.045190,0.066738
rf_d30,0.80,0.026572,2.003048e+05,2026-03-18 12:07:00.642142296,2026-02-16,95.0,65228.0,575.0,591.0,0.141791,0.138484,0.140118
lightgbm_d30,0.90,0.000000,0.000000e+00,2026-03-17 23:33:04.858723640,2026-02-16 00:00:00,0.0,65803.0,0.0,686.0,0.000000,0.000000,0.000000
logreg_d30,0.95,-0.920600,-6.939763e+06,2026-03-17 23:32:04.590365887,2026-02-16 00:00:00,31.0,65596.0,207.0,655.0,0.130252,0.045190,0.067100
rf_d30,0.90,0.112276,8.463687e+05,2026-03-17 23:32:59.349112749,2026-02-16 00:00:00,179.0,64459.0,1344.0,507.0,0.117531,0.260933,0.162064


In [9]:
pct_avoided.max(axis = 1, skipna = True, numeric_only = True)

0    0.039506
1    0.072108
dtype: float64

---
Main Runner here

In [ ]:
synthea_config_path = str(project_root / "config" / "synthea_config.json")

config_path_bq = str(project_root / "config" / "bigquery_config.json")
recipe_path = str(project_root / "config" / "bigquery_recipes.json")

io_config_path = str(project_root / "config" / "dictionary_io_config.json")
checks_dir = str(project_root / "sql" / "checks")

model_config_path = str(project_root / "config" / "model_config.json")
cost_config_path = str(project_root / "config" / "cost_config.json")

preproc = DataPreprocessor.from_config(model_config_path)

for profile_name in ['train', 'test']:

    """Run Synthea"""
    runner, run_params = SyntheaRunner.from_profile(
    config_path=synthea_config_path,
    profile_name=profile_name,
    )
    csv_dir = runner.run(**run_params)

    """Initialize BigQueryLoader and load raw_data onto BigQuery"""
    bq_loader, profile_cfg = BigQueryLoader.from_profile(
    config_path=config_path_bq,
    profile_name=profile_name,
    )
    bq_loader.load_profile_tables(profile_cfg)

    """Initialize BigQueryTransformer and create slim tables"""
    transformer, t_cfg = BigQueryTransformer.from_profile(
        config_path=config_path_bq,
        profile_name=profile_name,
    )
    recipes_id = 0
    transformer.run_query_sequence(
    recipe_path=recipe_path,
    recipes_id=recipes_id,
    project_root=str(project_root),
    )
    recipes_id += 1

    """Initialize DictionaryBuilder, build local dictionaries and load them"""
    builder = DictionaryBuilder(
    transformer=transformer,
    io_config_path=io_config_path,
    config_path_bq=config_path_bq,
    profile_name=profile_name,
    )
    builder.build_diagnoses_dictionary(profile_name)
    builder.build_procedures_dictionary(profile_name)
    builder.build_main_diagnoses(profile_name)
    bq_loader.load_dictionaries("dictionaries_dir")

    """Build and load careplans"""
    builder.build_careplans_related_diagnoses(profile_name)
    bq_loader.load_dictionaries("careplans_dir")

    """Build helpers, sanity check them, build and load related diagnoses, build index stay"""
    transformer.run_query_sequence(
    recipe_path=recipe_path,
    recipes_id=recipes_id,
    project_root=str(project_root),
    )
    recipes_id += 1

    transformer.run_helper_clinical_sanity_checks(checks_base_dir=checks_dir)
    transformer.run_helper_cost_sanity_checks(checks_base_dir=checks_dir)
    transformer.run_helper_utilization_sanity_checks(checks_base_dir=checks_dir)

    builder.build_related_diagnoses(profile_name)
    bq_loader.load_dictionaries("related_dir")

    transformer.run_query_sequence(
        recipe_path=recipe_path,
        recipes_id=recipes_id,
        project_root=str(project_root),
    )

    """Load and preprocess index stays"""
    X[profile_name], y[profile_name] = preproc.load_and_preprocess(
        transformer=transformer,
        force_query=False
    )

"""Initialize ModelConfigManager, populate models, initialize HyperparameterTuner and tune hyperparams"""
cfg_mgr = ModelConfigManager.from_config(model_config_path)
active_models = cfg_mgr.list_active_models()

tuner = HyperparameterTuner(config_mgr=cfg_mgr)

tuner.tune_models(X['train'], y['train'], target_cols="readmit_30d", cost_config_path=cost_config_path)
cfg_mgr.save()

"""Initialize ModelRegistry, build and fit models on optimal hyperparams"""
registry = ModelRegistry.from_config(model_config_path)
fitted_models = registry.fit_models(
    X=X['train'],
    y=y['train'],
    target_cols=["readmit_30d", "readmit_90d"],
    model_names=None  # all active models from config
)

"""Initialize Evaluator, get metrics from the models"""
evaluator = Evaluator(registry=registry, cfg_mgr=cfg_mgr)

eval_results = evaluator.evaluate_models(
    X=X['test'],
    y=y['test'],
    model_names=None        # all active   
)

eval_results.update(evaluator.build_threshold_metrics(eval_results["pred_values"]))

"""Initialize CostReducer and estimate saved spendings"""
cost_reducer = CostReducer.from_config(cost_config_path)
mapping, avoided, pct_avoided = cost_reducer.map_estimate_cost_reduction(
    pred_values=eval_results['pred_values'],
    df_thresholds=eval_results['thresholds'],
)

"""Save everything to local files, gotta put it into a CostReducer"""
pct_avoided.to_csv(r"D:\Python Projects\Hospital readmission risk\scripts\data\pct_avoided.csv")
avoided.to_csv(r"D:\Python Projects\Hospital readmission risk\scripts\data\avoided.csv")

report = evaluator.build_performance_report(
    pct_avoided=pct_avoided,
    avoided=avoided,
    threshold_metrics=eval_results['threshold_metrics'],
    model_file_dir=Path("models"),
    dataset_end_date=pd.Timestamp("2026-01-31"),
)


---

In [ ]:
# Cell 2: choose profile and run Synthea
config_path = str(project_root / "config" / "synthea_config.json")

# choose one of: "mock", "train", "test"
profile_name = "mock"

runner, run_params = SyntheaRunner.from_profile(
    config_path=config_path,
    profile_name=profile_name,
)

csv_dir = runner.run(**run_params)
csv_dir
"""DONE"""

INFO:pipeline.synthea_runner:Running Synthea with command: java -jar F:\Synthea\synthea-with-dependencies.jar -s 100 -cs 100 -p 1000 --exporter.csv.export=true --exporter.years_of_history=3 California
INFO:pipeline.synthea_runner:Synthea working directory: F:\Synthea
INFO:pipeline.synthea_runner:Synthea CSV directory: F:\Synthea\output\csv
INFO:pipeline.synthea_runner:Profile output_dir: D:\Python Projects\Hospital readmission risk\data\raw\Synthea\mock
INFO:pipeline.synthea_runner:Files to copy: {'patients.csv', 'conditions.csv', 'procedures.csv', 'careplans.csv', 'encounters.csv', 'organizations.csv', 'claims.csv', 'medications.csv'}
INFO:pipeline.synthea_runner:Delete source files after copy: True
INFO:pipeline.synthea_runner:Copying Synthea CSVs from F:\Synthea\output\csv to D:\Python Projects\Hospital readmission risk\data\raw\Synthea\mock
INFO:pipeline.synthea_runner:Copied F:\Synthea\output\csv\patients.csv -> D:\Python Projects\Hospital readmission risk\data\raw\Synthea\mock\pa

WindowsPath('D:/Python Projects/Hospital readmission risk/data/raw/Synthea/mock')

In [2]:
# Cell 3: Initialize BigQueryLoader and load raw_data onto BigQuery
config_path_bq = str(project_root / "config" / "bigquery_config.json")

profile_name = "mock"  # "train" or "test" as needed - keep it till the end of a pipeline

bq_loader, profile_cfg = BigQueryLoader.from_profile(
    config_path=config_path_bq,
    profile_name=profile_name,
)

bq_loader.load_profile_tables(profile_cfg)

INFO:pipeline.bq_loader:Loading CSVs from D:\Python Projects\Hospital readmission risk\data\raw\Synthea\mock into dataset healthcare-test-486920.raw_data
INFO:pipeline.bq_loader:Creating dataset: healthcare-test-486920.raw_data
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.raw_data.careplans
INFO:pipeline.bq_loader:Loaded 2258 rows into healthcare-test-486920.raw_data.careplans
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.raw_data.claims
INFO:pipeline.bq_loader:Loaded 50633 rows into healthcare-test-486920.raw_data.claims
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.raw_data.conditions
INFO:pipeline.bq_loader:Loaded 19574 rows into healthcare-test-486920.raw_data.conditions
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.raw_data.encounters
INFO:pipeline.bq_loader:Loaded 27766 rows into healthcare-test-486920.raw_data.encounters
INFO:pipeline.bq_loader

In [2]:
# Cell 4: Initialize BigQueryTransformer and create slim tables
recipe_path = str(project_root / "config" / "bigquery_recipes.json")
config_path_bq = str(project_root / "config" / "bigquery_config.json")

transformer, t_cfg = BigQueryTransformer.from_profile(config_path=config_path_bq)
recipes_id = 0
transformer.run_query_sequence(
    recipe_path=recipe_path,
    recipes_id=recipes_id,
    project_root=str(project_root),
)
recipes_id += 1

INFO:pipeline.bq_transformer:Creating dataset: healthcare-test-486920.data_slim
INFO:pipeline.bq_transformer:Creating dataset: healthcare-test-486920.helper_tables
INFO:pipeline.bq_transformer:Running query:
CREATE OR REPLACE TABLE healthcare-test-486920.data_slim.patients_slim
  CLUSTER BY id
AS
SELECT
  id,
  birthdate,
  deathdate,
  race,
  gender
FROM healthcare-test-486920.raw_data.patients
INFO:pipeline.bq_transformer:Query finished.
INFO:pipeline.bq_transformer:Running query:
CREATE OR REPLACE TABLE healthcare-test-486920.data_slim.encounters_slim
  CLUSTER BY id, patient, stop
AS (
  WITH
    end_date AS (
      SELECT MAX(stop) AS end_ts
      FROM healthcare-test-486920.raw_data.encounters
    ),
    p AS (
      SELECT
        id AS patient_id,
        deathdate
      FROM healthcare-test-486920.data_slim.patients_slim
    )
  SELECT
    e.id,
    e.start,
    e.stop,
    e.patient,
    e.organization,
    e.encounterclass,
    e.base_encounter_cost,
    e.total_claim_cost,

In [3]:
# Cell 5: Initialize DictionaryBuilder and build local dictionaries
io_config_path = str(project_root / "config" / "dictionary_io_config.json")

builder = DictionaryBuilder(
    transformer=transformer,
    io_config_path=io_config_path,
    config_path_bq=config_path_bq,
    profile_name=profile_name,
)

builder.build_diagnoses_dictionary(profile_name)
builder.build_procedures_dictionary(profile_name)
builder.build_main_diagnoses(profile_name)

INFO:pipeline.dictionary_builder:Building diagnoses dictionary for mock split
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
INFO:pipeline.dictionary_builder:Building procedures dictionary for mock split
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
INFO:pipeline.dictionary_builder:Building main_diagnoses for mock split
INFO:pipeline.dictionary_builder:Dictionary path: D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\mock_diagnoses_dictionary.csv
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint in

WindowsPath('D:/Python Projects/Hospital readmission risk/data/processed/dictionaries/mock_main_diagnoses.csv')

In [4]:
# Cell 6: Load dictionaries
bq_loader.load_dictionaries("dictionaries_dir")

INFO:pipeline.bq_loader:Dataset already exists: healthcare-test-486920.helper_tables
INFO:pipeline.bq_loader:Loading dictionary CSVs for profile 'mock' from D:\Python Projects\Hospital readmission risk\data\processed\dictionaries into dataset healthcare-test-486920.helper_tables with table prefix 'mock_'
INFO:pipeline.bq_loader:Loading dictionary file D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\mock_diagnoses_dictionary.csv into table healthcare-test-486920.helper_tables.mock_diagnoses_dictionary
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.helper_tables.mock_diagnoses_dictionary
INFO:pipeline.bq_loader:Loaded 276 rows into healthcare-test-486920.helper_tables.mock_diagnoses_dictionary
INFO:pipeline.bq_loader:Loading dictionary file D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\mock_main_diagnoses.csv into table healthcare-test-486920.helper_tables.mock_main_diagnoses
INFO:pipeline.bq_loader:Load

In [5]:
# Cell 7: Build and load careplans
builder.build_careplans_related_diagnoses(profile_name)
bq_loader.load_dictionaries("careplans_dir")

INFO:pipeline.dictionary_builder:Building careplans_related_diagnoses for mock split
INFO:pipeline.dictionary_builder:Dictionary path: D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\mock_diagnoses_dictionary.csv
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
INFO:pipeline.bq_loader:Dataset already exists: healthcare-test-486920.helper_tables
INFO:pipeline.bq_loader:Loading dictionary CSVs for profile 'mock' from D:\Python Projects\Hospital readmission risk\data\processed\careplans into dataset healthcare-test-486920.helper_tables with table prefix 'mock_'
INFO:pipeline.bq_loader:Loading dictionary file D:\Python Projects\Hospital readmission risk\data\processed\careplans\mock_careplans_related_encounters.csv into table healthcare-test-486920.helper_tables.mock_careplans_related_encounters
INFO:pi

In [ ]:
# Cell 8: Build helpers, sanity check them, build and load related diagnoses, build index stay
#recipes_id = 1
transformer.run_query_sequence(
    recipe_path=recipe_path,
    recipes_id=recipes_id,
    project_root=str(project_root),
)
recipes_id += 1

checks_dir = str(project_root / "sql" / "checks")
transformer.run_helper_clinical_sanity_checks(checks_base_dir=checks_dir)
transformer.run_helper_cost_sanity_checks(checks_base_dir=checks_dir)
transformer.run_helper_utilization_sanity_checks(checks_base_dir=checks_dir)

builder.build_related_diagnoses(profile_name)
bq_loader.load_dictionaries("related_dir")

transformer.run_query_sequence(
    recipe_path=recipe_path,
    recipes_id=recipes_id,
    project_root=str(project_root),
)

INFO:pipeline.bq_transformer:Running query:
CREATE OR REPLACE TABLE healthcare-test-486920.helper_tables.mock_helper_clinical
AS (
  WITH
    end_date AS (
      SELECT MAX(stop) AS end_ts
      FROM healthcare-test-486920.data_slim.mock_encounters_slim
    ),

    -- explode claims into one row per (stay, code) so we can see which of them are disorders
    claims_long AS (
      SELECT DISTINCT
        stay_id, code
      FROM
        (
          SELECT encounter AS stay_id, CAST(diagnosis1 AS INT64) AS code
          FROM healthcare-test-486920.data_slim.mock_claims_slim
          WHERE diagnosis1 IS NOT NULL
          UNION ALL
          SELECT encounter AS stay_id, CAST(diagnosis2 AS INT64) AS code
          FROM healthcare-test-486920.data_slim.mock_claims_slim
          WHERE diagnosis2 IS NOT NULL
          UNION ALL
          SELECT encounter AS stay_id, CAST(diagnosis3 AS INT64) AS code
          FROM healthcare-test-486920.data_slim.mock_claims_slim
          WHERE diagnosis3

In [10]:
#Cell 9: Initialize preprocessor, load and preprocess index stays
model_config_path = str(project_root / "config" / "model_config.json")

preproc = DataPreprocessor.from_config(model_config_path)

X[profile_name], y[profile_name] = preproc.load_and_preprocess(
    transformer=transformer,
    profile_name=profile_name,
    force_query=True,
)

d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
#Cell 10: Loop ends, Initialize ModelConfigManager, populate models, initialize HyperparameterTuner and tune hyperparams
cfg_mgr = ModelConfigManager.from_config(model_config_path)
active_models = cfg_mgr.list_active_models()

#if profile_name == "train":
tuner = HyperparameterTuner(config_mgr=cfg_mgr)
tuner.tune_models(X['train'], y['train'], target_cols="readmit_30d", cost_config_path=cost_config_path)

cfg_mgr.save()

d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=50. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 4 candidates, totalling 20 fits


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


logreg: best score is 0.24171775958616548
Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


rf: best score is 0.3737514078371348
Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[LightGBM] [Info] Number of positive: 36, number of negative: 1505
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000189 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1037
[LightGBM] [Info] Number of data points in the train set: 1541, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.023361 -> initscore=-3.733029
[LightGBM] [Info] Start training from score -3.733029
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

In [ ]:
#Cell 11: Initialize ModelRegistry, build and fit models on optimal hyperparams
registry = ModelRegistry.from_config(
    config_path=model_config_path,
    models_dir="models",
)
fitted_models = registry.fit_models(
    X=X["train"],
    y=y["train"],
    target_cols=["readmit_30d"],
    model_names=None  # all active models from config
)

In [ ]:
#Cell 12: Get metrics from the models
evaluator = Evaluator(registry=registry, cfg_mgr=cfg_mgr)
eval_results = evaluator.evaluate_models(
    X=X["test"],
    y=y["test"],
    model_names=None,        # all active
    suffix=profile_name,     # same suffix used in fit_models
)
eval_results.update(evaluator.build_threshold_metrics(eval_results["pred_values"]))

In [ ]:
#Cell 13: Build cost reduction estimations
cost_config_path = str(project_root / "config" / "cost_config.json")
cost_reducer = CostReducer.from_config(cost_config_path)

mapping, avoided, pct_avoided = cost_reducer.map_estimate_cost_reduction(
    df_test=X['test'],
    pred_values=eval_results['pred_values'],
    df_thresholds=eval_results['thresholds'],
)


---